# ONNX export
Image-level text labels only. Dummy results do not establish OCR accuracy or deployment performance.

Set `USE_ENERGY_METER = True` if you trained the meter example.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
assert (ROOT / 'config/cct_s_v2_global_plate_config.yaml').exists()
def run(*args): subprocess.run(list(args), check=True)

USE_ENERGY_METER = os.environ.get("CCT_ENERGY_METER", "0") == "1"
WORK = Path(os.environ.get('CCT_WORK', 'artifacts/energy_meter' if USE_ENERGY_METER else 'artifacts'))
TRAINING = Path(os.environ.get("CCT_OUTPUT", WORK if USE_ENERGY_METER else WORK / "training"))
PLATE_CONFIG = ('config/cct_s_energy_meter_config.yaml' if USE_ENERGY_METER
                else 'config/cct_s_v2_global_plate_config.yaml')
ANNOTATIONS = ('dataset/energy-meter/val_annotations.csv' if USE_ENERGY_METER
               else 'dummy_data/val_annotations.csv')


In [ ]:
checkpoints = list(TRAINING.rglob('best.keras'))
if not checkpoints:
    raise FileNotFoundError('Run 02_train.ipynb first.')
MODEL = str(max(checkpoints, key=lambda p: p.stat().st_mtime))
(WORK / 'onnx').mkdir(parents=True,exist_ok=True)
run('fast-plate-ocr','export','--model',MODEL,'--format','onnx','--plate-config-file',PLATE_CONFIG,'--save-dir',str(WORK / 'onnx'),'--dynamic-batch','--onnx-input-dtype','float32','--onnx-data-format','channels_first')